In [8]:
import pandas as pd
import sqlite3
from pathlib import Path

# Load rb3 data
rb3_dir = Path.home() / "Documents/code/options-backtest/data/B3/rb3_export"
bova_rb3 = pd.read_parquet(rb3_dir / "bova11_options_rb3.parquet")
petr_rb3 = pd.read_parquet(rb3_dir / "petr4_options_rb3.parquet")

# Load your DB data
db_path = Path.home() / "Documents/code/options-backtest/data/B3" / "b3_options.db"
conn = sqlite3.connect(db_path)  # Update path
bova_yours = pd.read_sql("SELECT * FROM options_data WHERE underlying = 'BOVA11'", conn)
petr_yours = pd.read_sql("SELECT * FROM options_data WHERE underlying = 'PETR4'", conn)
# conn.close()

In [5]:
# First, check the schema
print("=== Database schema ===")
schema = pd.read_sql("PRAGMA table_info(options_data)", conn)
print(schema[["name", "type"]])

# Load your data
bova_yours = pd.read_sql(
    """
    SELECT * FROM options_data 
    WHERE underlying = 'BOVA11'
""",
    conn,
)

petr_yours = pd.read_sql(
    """
    SELECT * FROM options_data
    WHERE underlying = 'PETR4'
""",
    conn,
)

print(f"\n=== Row counts ===")
print(f"BOVA11: rb3={len(bova_rb3):,}  yours={len(bova_yours):,}")
print(f"PETR4:  rb3={len(petr_rb3):,}  yours={len(petr_yours):,}")

print("\n=== Your DB columns ===")
print(bova_yours.columns.tolist())

print("\n=== Your DB sample ===")
print(bova_yours.head(3))

print("\n=== rb3 sample ===")
print(
    bova_rb3[
        [
            "refdate",
            "symbol",
            "strike_price",
            "maturity_date",
            "option_type",
            "close",
            "best_bid",
            "best_ask",
            "impvol",
            "delta",
        ]
    ].head(3)
)

conn.close()

=== Database schema ===
                  name     type
0                 date     TEXT
1           expiration     TEXT
2                 type     TEXT
3               strike     REAL
4           underlying     TEXT
5                 mark     REAL
6                  bid     REAL
7                  ask     REAL
8               volume  INTEGER
9                delta     REAL
10  implied_volatility     REAL

=== Row counts ===
BOVA11: rb3=454,651  yours=0
PETR4:  rb3=971,157  yours=0

=== Your DB columns ===
['date', 'expiration', 'type', 'strike', 'underlying', 'mark', 'bid', 'ask', 'volume', 'delta', 'implied_volatility']

=== Your DB sample ===
Empty DataFrame
Columns: [date, expiration, type, strike, underlying, mark, bid, ask, volume, delta, implied_volatility]
Index: []

=== rb3 sample ===
      refdate   symbol  strike_price maturity_date option_type  close  \
0  2017-01-05  BOVAA58          58.0    2017-01-16        call   2.30   
1  2017-01-05  BOVAA59          59.0    2017-01-16

In [10]:
# Make sure we're connecting to the right file
print(f"Connecting to: {db_path}")

conn = sqlite3.connect(db_path)

# List ALL tables
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print("\n=== All tables in database ===")
print(tables)

# Count total rows in options_data table
total = pd.read_sql("SELECT COUNT(*) as count FROM options_data", conn)
print(f"\n=== Total rows in 'options_data' table ===")
print(total)

# Check what underlyings actually exist
underlyings = pd.read_sql(
    "SELECT DISTINCT underlying FROM options_data LIMIT 100", conn
)
print(f"\n=== Distinct underlyings (first 100) ===")
print(underlyings)

# Try case-insensitive search
bova_test = pd.read_sql(
    """
    SELECT COUNT(*) as count 
    FROM options_data 
    WHERE underlying LIKE '%BOVA%'
""",
    conn,
)
print(f"\n=== Rows with 'BOVA' anywhere in underlying ===")
print(bova_test)

# Sample ANY rows to see the data structure
sample = pd.read_sql("SELECT * FROM options_data LIMIT 5", conn)
print(f"\n=== First 5 rows from options_data table ===")
print(sample)

conn.close()

Connecting to: /Users/oliversteiner/Documents/code/options-backtest/data/B3/b3_options.db

=== All tables in database ===
           name
0  options_data

=== Total rows in 'options_data' table ===
     count
0  1100254

=== Distinct underlyings (first 100) ===
  underlying
0      petr4
1     bova11

=== Rows with 'BOVA' anywhere in underlying ===
    count
0  357643

=== First 5 rows from options_data table ===
         date  expiration  type  strike underlying  mark   bid   ask  volume  \
0  2015-01-02  2015-01-19  call    48.0     bova11  0.78   NaN  None    4200   
1  2015-01-02  2015-01-19  call    49.0     bova11  0.42   NaN  None   12000   
2  2015-01-02  2015-01-19  call    50.0     bova11  0.23   NaN  None   12010   
3  2015-01-02  2015-01-19  call    51.0     bova11  0.12  0.05  None   20700   
4  2015-01-02  2015-02-09  call    51.0     bova11  0.45   NaN  None    4000   

      delta  implied_volatility  
0  0.434347            0.245191  
1  0.284137            0.239046  
2

In [12]:
# Load your database (correct table name and lowercase underlyings)
conn = sqlite3.connect(db_path)

bova_yours = pd.read_sql(
    """
    SELECT * FROM options_data 
    WHERE underlying = 'bova11'
""",
    conn,
)

petr_yours = pd.read_sql(
    """
    SELECT * FROM options_data
    WHERE underlying = 'petr4'
""",
    conn,
)

conn.close()

print(f"=== Row counts ===")
print(f"BOVA11: rb3={len(bova_rb3):,}  yours={len(bova_yours):,}")
print(f"PETR4:  rb3={len(petr_rb3):,}  yours={len(petr_yours):,}")

# Prepare for merge: standardize column names and types
# rb3 uses: refdate, strike_price, maturity_date, option_type, close
# yours uses: date, strike, expiration, type, mark

# Convert dates to same format
bova_rb3["date_str"] = bova_rb3["refdate"].astype(str)
bova_rb3["exp_str"] = bova_rb3["maturity_date"].astype(str)

bova_yours["date_str"] = bova_yours["date"]
bova_yours["exp_str"] = bova_yours["expiration"]

# Merge on (date, strike, expiration, option_type)
bova_merged = bova_rb3.merge(
    bova_yours,
    left_on=["date_str", "strike_price", "exp_str", "option_type"],
    right_on=["date_str", "strike", "exp_str", "type"],
    how="inner",
    suffixes=("_rb3", "_yours"),
)

print(f"\n=== BOVA11 Merge Results ===")
print(f"Matched rows: {len(bova_merged):,}")
print(f"Match rate: {len(bova_merged)/min(len(bova_rb3), len(bova_yours))*100:.1f}%")

if len(bova_merged) > 0:
    # Compare close prices (rb3: 'close', yours: 'mark')
    price_diff = (bova_merged["close"] - bova_merged["mark"]).abs()

    print(f"\n=== Close Price Comparison (rb3.close vs yours.mark) ===")
    print(f"Mean absolute diff: R$ {price_diff.mean():.4f}")
    print(f"Median diff: R$ {price_diff.median():.4f}")
    print(f"Max diff: R$ {price_diff.max():.4f}")
    print(f"% within R$0.01: {(price_diff < 0.01).mean()*100:.1f}%")
    print(f"% within R$0.05: {(price_diff < 0.05).mean()*100:.1f}%")

    # Compare IVs
    iv_diff = (bova_merged["impvol"] - bova_merged["implied_volatility"]).abs()

    print(f"\n=== IV Comparison ===")
    print(f"Mean absolute diff: {iv_diff.mean():.4f}")
    print(f"Median diff: {iv_diff.median():.4f}")
    print(f"% within 0.02 (2 vol points): {(iv_diff < 0.02).mean()*100:.1f}%")
    print(f"% within 0.05 (5 vol points): {(iv_diff < 0.05).mean()*100:.1f}%")

    # Compare deltas
    delta_diff = (bova_merged["delta_rb3"] - bova_merged["delta_yours"]).abs()

    print(f"\n=== Delta Comparison ===")
    print(f"Mean absolute diff: {delta_diff.mean():.4f}")
    print(f"Median diff: {delta_diff.median():.4f}")
    print(f"% within 0.05: {(delta_diff < 0.05).mean()*100:.1f}%")

    # Show worst mismatches
    print(f"\n=== Top 5 Close Price Mismatches ===")
    worst = bova_merged.nlargest(5, price_diff.values)
    print(
        worst[
            [
                "date_str",
                "strike_price",
                "exp_str",
                "option_type",
                "close",
                "mark",
                "impvol",
                "implied_volatility",
            ]
        ].to_string()
    )

else:
    print("\n❌ WARNING: Zero rows matched!")
    print("Checking date ranges...")
    print(f"rb3 dates: {bova_rb3['refdate'].min()} to {bova_rb3['refdate'].max()}")
    print(f"yours dates: {bova_yours['date'].min()} to {bova_yours['date'].max()}")
    print("\nSample rb3 rows:")
    print(bova_rb3[["refdate", "strike_price", "maturity_date", "option_type"]].head(3))
    print("\nSample your rows:")
    print(bova_yours[["date", "strike", "expiration", "type"]].head(3))

=== Row counts ===
BOVA11: rb3=454,651  yours=357,643
PETR4:  rb3=971,157  yours=742,611

=== BOVA11 Merge Results ===
Matched rows: 304,173
Match rate: 85.0%

=== Close Price Comparison (rb3.close vs yours.mark) ===
Mean absolute diff: R$ 8.4069
Median diff: R$ 0.0050
Max diff: R$ 499997.6950
% within R$0.01: 55.4%
% within R$0.05: 70.2%

=== IV Comparison ===
Mean absolute diff: 0.0347
Median diff: 0.0147
% within 0.02 (2 vol points): 47.0%
% within 0.05 (5 vol points): 66.6%

=== Delta Comparison ===
Mean absolute diff: 0.0186
Median diff: 0.0086
% within 0.05: 72.1%

=== Top 5 Close Price Mismatches ===


KeyError: np.float64(0.0)

In [13]:
# The Top 5 mismatches didn't print - let's get them manually
print(f"\n=== Top 10 Close Price Mismatches ===")
bova_merged["price_diff"] = (bova_merged["close"] - bova_merged["mark"]).abs()
worst = bova_merged.nlargest(10, "price_diff")
print(
    worst[
        [
            "date_str",
            "strike_price",
            "exp_str",
            "option_type",
            "close",
            "mark",
            "volume_rb3",
            "volume_yours",
            "impvol",
            "implied_volatility",
        ]
    ].to_string()
)

# Check if outliers are concentrated in specific conditions
print(f"\n=== Outlier Analysis ===")
big_diff = bova_merged[bova_merged["price_diff"] > 1.0]  # >R$1 difference
print(
    f"Rows with >R$1 diff: {len(big_diff):,} ({len(big_diff)/len(bova_merged)*100:.1f}%)"
)

if len(big_diff) > 0:
    print(f"\nOption types in outliers:")
    print(big_diff["option_type"].value_counts())

    print(f"\nMoneyness in outliers (strike/underlying):")
    print(big_diff["strike_price"].describe())

    print(f"\nVolume in outliers (rb3):")
    print(big_diff["volume_rb3"].describe())

    print(f"\nVolume in outliers (yours):")
    print(big_diff["volume_yours"].describe())

    # Check for zero/null volumes (illiquid options with stale prices)
    zero_vol = big_diff[big_diff["volume_rb3"] == 0]
    print(
        f"\nOutliers with zero volume (rb3): {len(zero_vol)} ({len(zero_vol)/len(big_diff)*100:.1f}% of outliers)"
    )

# Now check the GOOD matches
print(f"\n=== Well-Matched Rows (diff < R$0.01) ===")
good = bova_merged[bova_merged["price_diff"] < 0.01]
print(f"Count: {len(good):,}")
print(f"\nSample of good matches:")
print(
    good[
        [
            "date_str",
            "strike_price",
            "option_type",
            "close",
            "mark",
            "impvol",
            "implied_volatility",
        ]
    ]
    .head(10)
    .to_string()
)


=== Top 10 Close Price Mismatches ===
          date_str  strike_price     exp_str option_type  close        mark  volume_rb3  volume_yours    impvol  implied_volatility
36774   2020-10-02          94.0  2020-10-19         put   3.95  500001.645   227796.26         70240  0.233288                 NaN
35797   2020-09-18         100.0  2020-09-21         put   5.40  500001.645   147482.30         29720  0.556723                 NaN
36311   2020-09-21         100.0  2020-10-19         put   7.27  500003.305    13880.30          1820  0.356075                 NaN
36504   2020-09-28         100.0  2020-10-19         put   7.99  500003.035     5211.00           700       NaN                 NaN
36315   2020-09-21         103.0  2020-10-19         put  10.84  500003.245     1084.00           100  0.508778                 NaN
105861  2024-06-20         123.0  2024-06-21         put   5.96     726.750  6419590.73       1146679  0.537490                 NaN
104672  2024-06-18         123.0  202